# End-to-End Lakehouse & BI Pipeline: Databricks Medallion Architecture → Tableau Executive Dashboard

**Architecture Overview:**
1. **Bronze Layer (`raw_orders`)**: Ingest raw transactional CRM & ERP sales streams into Delta Lake.
2. **Silver Layer (`cleaned_orders`)**: Standardize currency (EUR), deduplicate records, validate schema, and compute net margins using PySpark.
3. **Gold Layer (`executive_sales_gold`)**: Aggregate business KPIs, YoY metrics, and regional performance marts optimized for **Tableau Public / Tableau Cloud**.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("ExecutiveSalesLakehouse").getOrCreate()

In [2]:
# 1. Bronze -> Silver Transformation: Clean & enrich sales transactions
df_bronze = spark.read.option("header", "true").option("inferSchema", "true").csv("../data/executive_sales_gold.csv")

df_silver = (
    df_bronze
    .dropDuplicates(["Order_ID"])
    .filter(F.col("Revenue_EUR") > 0)
    .withColumn("Order_Date", F.to_date(F.col("Order_Date")))
    .withColumn("Net_Profit_Ratio", F.round(F.col("Profit_EUR") / F.col("Revenue_EUR"), 4))
    .withColumn("Target_Achievement_Ratio", F.round(F.col("Revenue_EUR") / F.col("Target_Revenue_EUR"), 4))
)

df_silver.createOrReplaceTempView("silver_sales")

In [3]:
# 2. Gold Layer SQL Mart for Executive Tableau Reporting
df_gold_summary = spark.sql("""
    SELECT
        Year,
        Quarter,
        Region,
        Country,
        Customer_Segment,
        Product_Category,
        COUNT(DISTINCT Order_ID) AS Total_Orders,
        ROUND(SUM(Revenue_EUR), 2) AS Total_Revenue_EUR,
        ROUND(SUM(Profit_EUR), 2) AS Total_Profit_EUR,
        ROUND(AVG(Profit_Margin_Pct), 2) AS Avg_Margin_Pct,
        ROUND(SUM(Target_Revenue_EUR), 2) AS Total_Target_EUR
    FROM silver_sales
    GROUP BY Year, Quarter, Region, Country, Customer_Segment, Product_Category
    ORDER BY Year DESC, Quarter DESC, Total_Revenue_EUR DESC
""")

df_gold_summary.show(10, truncate=False)